# ACDADA — Notebook 01: Data Preprocessing & Feature Engineering

**Autonomous Cyber Deception & Adaptive Defense Agent**

This notebook handles:
1. Loading raw datasets (CICIDS2017, UNSW-NB15, Bot-IoT)
2. Exploratory Data Analysis (EDA)
3. Data cleaning (missing values, duplicates, infinity handling)
4. Feature engineering (encoding, normalization, selection)
5. Train/Val/Test splitting
6. Saving processed data for downstream notebooks

## Dataset Download Links

Download and place raw CSVs into `../data/raw/`

In [ ]:
# ============================================================
# DATASET DOWNLOAD LINKS
# ============================================================
#
# 1. CIC-IDS-2017 (Network Intrusion Dataset)
#    https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset
#    -> Download and extract to: ../data/raw/cicids2017/
#
# 2. UNSW-NB15
#    https://research.unsw.edu.au/projects/unsw-nb15-dataset
#    https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15
#    -> Download and extract to: ../data/raw/unsw_nb15/
#
# 3. Bot-IoT Dataset
#    https://research.unsw.edu.au/projects/bot-iot-dataset
#    https://www.kaggle.com/datasets/vigneshvenkateswaran/bot-iot-dataset
#    -> Download and extract to: ../data/raw/bot_iot/
#
# BETH Dataset (supplementary):
#    https://www.kaggle.com/datasets/katehighnam/beth-dataset
#    -> Download and extract to: ../data/raw/beth/
#
# ============================================================
# DIRECTORY STRUCTURE AFTER DOWNLOAD:
# ../data/raw/
#   ├── cicids2017/        (CSV files from CICIDS2017)
#   ├── unsw_nb15/         (CSV files from UNSW-NB15)
#   ├── bot_iot/           (CSV files from Bot-IoT)
#   └── beth/              (CSV files from BETH)
# ============================================================

## 1. Imports & Configuration

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, LabelEncoder, RobustScaler
)
from sklearn.feature_selection import (
    mutual_info_classif, SelectKBest, VarianceThreshold
)
from sklearn.utils import resample
import joblib
import gc

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# Paths
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'Raw data directory: {RAW_DIR.resolve()}')
print(f'Processed data directory: {PROCESSED_DIR.resolve()}')

---
## 2. Load CICIDS2017 Dataset

In [ ]:
def load_cicids2017(data_dir: Path) -> pd.DataFrame:
    """
    Load all CICIDS2017 CSV files and concatenate into a single DataFrame.
    Handles the common issues: whitespace in column names, mixed dtypes.
    """
    cicids_path = data_dir / 'cicids2017'
    csv_files = sorted(cicids_path.glob('*.csv'))
    
    if not csv_files:
        print(f'[WARNING] No CSV files found in {cicids_path}')
        return pd.DataFrame()
    
    print(f'Found {len(csv_files)} CICIDS2017 files:')
    for f in csv_files:
        print(f'  - {f.name} ({f.stat().st_size / 1e6:.1f} MB)')
    
    dfs = []
    for f in csv_files:
        try:
            df = pd.read_csv(f, low_memory=False, encoding='utf-8')
            # Strip whitespace from column names
            df.columns = df.columns.str.strip()
            dfs.append(df)
            print(f'  Loaded {f.name}: {df.shape}')
        except Exception as e:
            print(f'  [ERROR] Failed to load {f.name}: {e}')
    
    if not dfs:
        return pd.DataFrame()
    
    combined = pd.concat(dfs, ignore_index=True)
    print(f'\nCombined CICIDS2017 shape: {combined.shape}')
    return combined

df_cicids = load_cicids2017(RAW_DIR)
if not df_cicids.empty:
    display(df_cicids.head())

## 3. Load UNSW-NB15 Dataset

In [ ]:
def load_unsw_nb15(data_dir: Path) -> pd.DataFrame:
    """
    Load UNSW-NB15 dataset. Handles both single-file and multi-file variants.
    The dataset has predefined train/test splits — we load both and combine.
    """
    unsw_path = data_dir / 'unsw_nb15'
    csv_files = sorted(unsw_path.glob('*.csv'))
    
    if not csv_files:
        print(f'[WARNING] No CSV files found in {unsw_path}')
        return pd.DataFrame()
    
    print(f'Found {len(csv_files)} UNSW-NB15 files:')
    for f in csv_files:
        print(f'  - {f.name} ({f.stat().st_size / 1e6:.1f} MB)')
    
    dfs = []
    for f in csv_files:
        try:
            df = pd.read_csv(f, low_memory=False, encoding='utf-8')
            df.columns = df.columns.str.strip()
            dfs.append(df)
            print(f'  Loaded {f.name}: {df.shape}')
        except Exception as e:
            print(f'  [ERROR] Failed to load {f.name}: {e}')
    
    if not dfs:
        return pd.DataFrame()
    
    combined = pd.concat(dfs, ignore_index=True)
    print(f'\nCombined UNSW-NB15 shape: {combined.shape}')
    return combined

df_unsw = load_unsw_nb15(RAW_DIR)
if not df_unsw.empty:
    display(df_unsw.head())

## 4. Load Bot-IoT Dataset

In [ ]:
def load_bot_iot(data_dir: Path) -> pd.DataFrame:
    """
    Load Bot-IoT dataset CSV files.
    Bot-IoT is heavily imbalanced — mostly attack traffic.
    """
    bot_path = data_dir / 'bot_iot'
    csv_files = sorted(bot_path.glob('*.csv'))
    
    if not csv_files:
        print(f'[WARNING] No CSV files found in {bot_path}')
        return pd.DataFrame()
    
    print(f'Found {len(csv_files)} Bot-IoT files:')
    for f in csv_files:
        print(f'  - {f.name} ({f.stat().st_size / 1e6:.1f} MB)')
    
    dfs = []
    for f in csv_files:
        try:
            df = pd.read_csv(f, low_memory=False, encoding='utf-8')
            df.columns = df.columns.str.strip()
            dfs.append(df)
            print(f'  Loaded {f.name}: {df.shape}')
        except Exception as e:
            print(f'  [ERROR] Failed to load {f.name}: {e}')
    
    if not dfs:
        return pd.DataFrame()
    
    combined = pd.concat(dfs, ignore_index=True)
    print(f'\nCombined Bot-IoT shape: {combined.shape}')
    return combined

df_botiot = load_bot_iot(RAW_DIR)
if not df_botiot.empty:
    display(df_botiot.head())

---
## 5. Exploratory Data Analysis (EDA)

We perform EDA on each dataset independently before unifying them.

In [ ]:
def dataset_overview(df: pd.DataFrame, name: str):
    """Print comprehensive overview of a dataset."""
    print(f'\n{"="*60}')
    print(f'  DATASET OVERVIEW: {name}')
    print(f'{"="*60}')
    print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
    print(f'\nColumn dtypes:')
    print(df.dtypes.value_counts())
    print(f'\nMissing values:')
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(missing.sort_values(ascending=False))
    else:
        print('  No missing values!')
    print(f'\nDuplicate rows: {df.duplicated().sum():,}')
    print(f'\nNumeric columns with inf values:')
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts = np.isinf(df[numeric_cols]).sum()
    inf_counts = inf_counts[inf_counts > 0]
    if len(inf_counts) > 0:
        print(inf_counts)
    else:
        print('  No infinite values!')
    print(f'\nFirst 5 column names: {list(df.columns[:5])}')
    print(f'Last 5 column names: {list(df.columns[-5:])}')
    return

# Run EDA on each available dataset
for df, name in [(df_cicids, 'CICIDS2017'), (df_unsw, 'UNSW-NB15'), (df_botiot, 'Bot-IoT')]:
    if not df.empty:
        dataset_overview(df, name)

In [ ]:
def plot_label_distribution(df: pd.DataFrame, label_col: str, name: str, top_n: int = 15):
    """Plot class/label distribution."""
    if label_col not in df.columns:
        print(f'[WARNING] Column "{label_col}" not found in {name}')
        return
    
    counts = df[label_col].value_counts().head(top_n)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'{name} — Label Distribution ({label_col})', fontsize=14, fontweight='bold')
    
    # Bar plot
    colors = sns.color_palette('viridis', len(counts))
    axes[0].barh(counts.index.astype(str), counts.values, color=colors)
    axes[0].set_xlabel('Count')
    axes[0].set_title('Class Counts (Top {})'.format(top_n))
    for i, v in enumerate(counts.values):
        axes[0].text(v + counts.values.max() * 0.01, i, f'{v:,}', va='center', fontsize=9)
    
    # Pie chart (log scale for readability)
    axes[1].pie(counts.values, labels=counts.index.astype(str), autopct='%1.1f%%',
                colors=colors, startangle=90)
    axes[1].set_title('Class Proportions')
    
    plt.tight_layout()
    plt.show()
    
    # Print counts
    print(f'\n{name} class counts:')
    print(df[label_col].value_counts())

# CICIDS2017 label column is typically 'Label'
if not df_cicids.empty:
    label_col_cicids = 'Label' if 'Label' in df_cicids.columns else df_cicids.columns[-1]
    plot_label_distribution(df_cicids, label_col_cicids, 'CICIDS2017')

# UNSW-NB15 label column: 'attack_cat' (multi-class) and 'label' (binary)
if not df_unsw.empty:
    label_col_unsw = 'attack_cat' if 'attack_cat' in df_unsw.columns else df_unsw.columns[-1]
    plot_label_distribution(df_unsw, label_col_unsw, 'UNSW-NB15')

# Bot-IoT label column: 'category' or 'attack'
if not df_botiot.empty:
    label_col_botiot = 'category' if 'category' in df_botiot.columns else (
        'attack' if 'attack' in df_botiot.columns else df_botiot.columns[-1]
    )
    plot_label_distribution(df_botiot, label_col_botiot, 'Bot-IoT')

In [ ]:
def plot_feature_distributions(df: pd.DataFrame, name: str, n_features: int = 12):
    """Plot distributions of numeric features."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 0:
        print(f'No numeric columns in {name}')
        return
    
    # Select a subset to plot
    cols_to_plot = numeric_cols[:n_features]
    n_rows = (len(cols_to_plot) + 3) // 4
    
    fig, axes = plt.subplots(n_rows, 4, figsize=(20, 4 * n_rows))
    fig.suptitle(f'{name} — Feature Distributions (first {n_features})', fontsize=14, fontweight='bold')
    axes = axes.flatten()
    
    for i, col in enumerate(cols_to_plot):
        data = df[col].replace([np.inf, -np.inf], np.nan).dropna()
        if data.nunique() > 1:
            axes[i].hist(data.clip(data.quantile(0.01), data.quantile(0.99)),
                        bins=50, color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.3)
        axes[i].set_title(col, fontsize=10)
        axes[i].tick_params(labelsize=8)
    
    # Hide empty subplots
    for j in range(len(cols_to_plot), len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    plt.show()

for df, name in [(df_cicids, 'CICIDS2017'), (df_unsw, 'UNSW-NB15'), (df_botiot, 'Bot-IoT')]:
    if not df.empty:
        plot_feature_distributions(df, name)

In [ ]:
def plot_correlation_matrix(df: pd.DataFrame, name: str, top_n: int = 20):
    """Plot correlation matrix for top N numeric features (by variance)."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df_num = df[numeric_cols].replace([np.inf, -np.inf], np.nan).dropna(axis=1, how='all')
    
    # Select top N by variance
    variances = df_num.var().sort_values(ascending=False)
    top_cols = variances.head(top_n).index.tolist()
    
    corr = df_num[top_cols].corr()
    
    fig, ax = plt.subplots(figsize=(14, 12))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=False,
                square=True, linewidths=0.5, ax=ax,
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{name} — Correlation Matrix (Top {top_n} by Variance)', fontsize=13)
    plt.tight_layout()
    plt.show()

for df, name in [(df_cicids, 'CICIDS2017'), (df_unsw, 'UNSW-NB15'), (df_botiot, 'Bot-IoT')]:
    if not df.empty:
        plot_correlation_matrix(df, name)

---
## 6. Data Cleaning Pipeline

Common issues across all datasets:
- Missing values (NaN)
- Infinite values (±inf)
- Duplicate rows
- Inconsistent column naming
- Non-numeric entries in numeric columns

In [ ]:
class DataCleaner:
    """
    Unified data cleaning pipeline for network intrusion datasets.
    Handles: column normalization, inf/nan removal, dtype fixes,
    duplicate removal, and label standardization.
    """
    
    def __init__(self, verbose: bool = True):
        self.verbose = verbose
        self.cleaning_log = []
    
    def _log(self, msg: str):
        self.cleaning_log.append(msg)
        if self.verbose:
            print(f'  [CLEAN] {msg}')
    
    def normalize_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Standardize column names: lowercase, strip, replace spaces."""
        original_cols = list(df.columns)
        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
            .str.replace(' ', '_', regex=False)
            .str.replace('/', '_', regex=False)
            .str.replace('-', '_', regex=False)
        )
        changed = sum(1 for a, b in zip(original_cols, df.columns) if a != b)
        self._log(f'Normalized {changed} column names')
        return df
    
    def remove_duplicates(self, df: pd.DataFrame) -> pd.DataFrame:
        """Remove duplicate rows."""
        n_before = len(df)
        df = df.drop_duplicates().reset_index(drop=True)
        n_removed = n_before - len(df)
        self._log(f'Removed {n_removed:,} duplicate rows ({n_removed/max(n_before,1)*100:.1f}%)')
        return df
    
    def handle_infinity(self, df: pd.DataFrame) -> pd.DataFrame:
        """Replace inf/-inf with NaN, then impute."""
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        inf_count = np.isinf(df[numeric_cols]).sum().sum()
        df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
        self._log(f'Replaced {inf_count:,} infinite values with NaN')
        return df
    
    def handle_missing(self, df: pd.DataFrame, strategy: str = 'median') -> pd.DataFrame:
        """Handle missing values in numeric columns."""
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        n_missing_before = df[numeric_cols].isnull().sum().sum()
        
        if strategy == 'median':
            df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
        elif strategy == 'mean':
            df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
        elif strategy == 'zero':
            df[numeric_cols] = df[numeric_cols].fillna(0)
        elif strategy == 'drop':
            df = df.dropna(subset=numeric_cols).reset_index(drop=True)
        
        # Handle categorical missing values
        cat_cols = df.select_dtypes(include=['object', 'category']).columns
        for col in cat_cols:
            df[col] = df[col].fillna('unknown')
        
        n_missing_after = df.isnull().sum().sum()
        self._log(f'Missing values: {n_missing_before:,} → {n_missing_after:,} (strategy={strategy})')
        return df
    
    def fix_dtypes(self, df: pd.DataFrame) -> pd.DataFrame:
        """Convert columns to appropriate dtypes to save memory."""
        mem_before = df.memory_usage(deep=True).sum() / 1e6
        
        # Downcast numeric columns
        for col in df.select_dtypes(include=['float64']).columns:
            df[col] = pd.to_numeric(df[col], downcast='float')
        for col in df.select_dtypes(include=['int64']).columns:
            df[col] = pd.to_numeric(df[col], downcast='integer')
        
        mem_after = df.memory_usage(deep=True).sum() / 1e6
        self._log(f'Memory: {mem_before:.1f} MB → {mem_after:.1f} MB ({(1 - mem_after/mem_before)*100:.1f}% reduction)')
        return df
    
    def remove_constant_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Remove columns with zero variance (constant)."""
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        n_before = len(numeric_cols)
        constant_cols = [col for col in numeric_cols if df[col].nunique() <= 1]
        if constant_cols:
            df = df.drop(columns=constant_cols)
        self._log(f'Removed {len(constant_cols)} constant features: {constant_cols}')
        return df
    
    def clean(self, df: pd.DataFrame, name: str = 'dataset') -> pd.DataFrame:
        """Run the full cleaning pipeline."""
        print(f'\n{"="*50}')
        print(f'  Cleaning: {name}')
        print(f'  Input shape: {df.shape}')
        print(f'{"="*50}')
        
        df = self.normalize_columns(df)
        df = self.remove_duplicates(df)
        df = self.handle_infinity(df)
        df = self.handle_missing(df, strategy='median')
        df = self.remove_constant_features(df)
        df = self.fix_dtypes(df)
        
        print(f'  Output shape: {df.shape}')
        return df

# Initialize cleaner
cleaner = DataCleaner(verbose=True)

# Clean each dataset
if not df_cicids.empty:
    df_cicids = cleaner.clean(df_cicids, 'CICIDS2017')

if not df_unsw.empty:
    df_unsw = cleaner.clean(df_unsw, 'UNSW-NB15')

if not df_botiot.empty:
    df_botiot = cleaner.clean(df_botiot, 'Bot-IoT')

---
## 7. Label Standardization

Create unified binary and multi-class labels across all datasets:
- **Binary**: `0 = Benign`, `1 = Attack`
- **Multi-class**: Unified attack categories (DDoS, DoS, Brute Force, Botnet, Port Scan, Injection, Other)

In [ ]:
# ============================================================
# CICIDS2017 Label Mapping
# ============================================================
CICIDS_LABEL_MAP = {
    'BENIGN': 'Benign',
    'DDoS': 'DDoS',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'Heartbleed': 'DoS',
    'PortScan': 'PortScan',
    'FTP-Patator': 'BruteForce',
    'SSH-Patator': 'BruteForce',
    'Bot': 'Botnet',
    'Web Attack \x96 Brute Force': 'BruteForce',
    'Web Attack \x96 XSS': 'Injection',
    'Web Attack \x96 Sql Injection': 'Injection',
    'Infiltration': 'Other',
}

# ============================================================
# UNSW-NB15 Label Mapping
# ============================================================
UNSW_LABEL_MAP = {
    'Normal': 'Benign',
    'Fuzzers': 'Other',
    'Analysis': 'Other',
    'Backdoor': 'Other',
    'Backdoors': 'Other',
    'DoS': 'DoS',
    'Exploits': 'Injection',
    'Generic': 'Other',
    'Reconnaissance': 'PortScan',
    'Shellcode': 'Injection',
    'Worms': 'Other',
}

# ============================================================
# Bot-IoT Label Mapping
# ============================================================
BOTIOT_LABEL_MAP = {
    'Normal': 'Benign',
    'DDoS': 'DDoS',
    'DoS': 'DoS',
    'Reconnaissance': 'PortScan',
    'Theft': 'Other',
}


def standardize_labels(df: pd.DataFrame, label_col: str, label_map: dict, name: str) -> pd.DataFrame:
    """
    Standardize labels using a mapping dictionary.
    Creates: 'attack_category' (multi-class) and 'is_attack' (binary).
    """
    if label_col not in df.columns:
        print(f'[WARNING] Column "{label_col}" not in {name}. Available: {list(df.columns[-5:])}')
        return df
    
    # Map labels
    original_labels = df[label_col].str.strip().unique()
    df['attack_category'] = df[label_col].str.strip().map(label_map).fillna('Other')
    
    # Binary label
    df['is_attack'] = (df['attack_category'] != 'Benign').astype(int)
    
    # Report unmapped labels
    mapped_labels = set(label_map.keys())
    unmapped = set(original_labels) - mapped_labels
    if unmapped:
        print(f'  [INFO] Unmapped labels in {name} (mapped to Other): {unmapped}')
    
    print(f'\n{name} — Standardized Labels:')
    print(f'  Binary: {df["is_attack"].value_counts().to_dict()}')
    print(f'  Multi-class: {df["attack_category"].value_counts().to_dict()}')
    
    return df

# Apply label standardization
if not df_cicids.empty:
    label_col = 'label' if 'label' in df_cicids.columns else 'Label'
    # After column normalization it's lowercase
    if 'label' in df_cicids.columns:
        df_cicids = standardize_labels(df_cicids, 'label', CICIDS_LABEL_MAP, 'CICIDS2017')

if not df_unsw.empty:
    label_col = 'attack_cat' if 'attack_cat' in df_unsw.columns else 'attack_category'
    if label_col in df_unsw.columns:
        df_unsw = standardize_labels(df_unsw, label_col, UNSW_LABEL_MAP, 'UNSW-NB15')

if not df_botiot.empty:
    label_col = 'category' if 'category' in df_botiot.columns else 'attack'
    if label_col in df_botiot.columns:
        df_botiot = standardize_labels(df_botiot, label_col, BOTIOT_LABEL_MAP, 'Bot-IoT')

---
## 8. Feature Engineering

In [ ]:
class FeatureEngineer:
    """
    Feature engineering pipeline:
    - Encode categoricals (Label Encoding for tree models, one-hot for DL)
    - Scale numeric features
    - Select top-K features by mutual information
    - Remove highly correlated features
    """
    
    def __init__(self, target_binary: str = 'is_attack', target_multi: str = 'attack_category'):
        self.target_binary = target_binary
        self.target_multi = target_multi
        self.label_encoders = {}
        self.scaler = None
        self.selected_features = None
        self.feature_importance = None
    
    def encode_categoricals(self, df: pd.DataFrame) -> pd.DataFrame:
        """Label-encode categorical features (excluding target columns)."""
        cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        exclude = [self.target_binary, self.target_multi]
        cat_cols = [c for c in cat_cols if c not in exclude]
        
        for col in cat_cols:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            self.label_encoders[col] = le
            print(f'  Encoded "{col}": {len(le.classes_)} classes')
        
        return df
    
    def remove_highly_correlated(self, df: pd.DataFrame, threshold: float = 0.95) -> pd.DataFrame:
        """Remove features with correlation > threshold."""
        exclude = [self.target_binary, self.target_multi]
        numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
        
        corr_matrix = df[numeric_cols].corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
        
        if to_drop:
            df = df.drop(columns=to_drop)
            print(f'  Removed {len(to_drop)} highly correlated features (r > {threshold})')
        
        return df
    
    def select_features_mi(self, df: pd.DataFrame, k: int = 50) -> pd.DataFrame:
        """Select top-K features using mutual information."""
        exclude = [self.target_binary, self.target_multi]
        feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
        
        if len(feature_cols) <= k:
            print(f'  Feature selection: {len(feature_cols)} features ≤ k={k}, keeping all.')
            self.selected_features = feature_cols
            return df
        
        X = df[feature_cols].values
        y = df[self.target_binary].values
        
        # Mutual Information
        mi_scores = mutual_info_classif(X, y, random_state=RANDOM_STATE, n_neighbors=5)
        mi_df = pd.DataFrame({'feature': feature_cols, 'mi_score': mi_scores})
        mi_df = mi_df.sort_values('mi_score', ascending=False)
        
        self.feature_importance = mi_df
        top_features = mi_df.head(k)['feature'].tolist()
        self.selected_features = top_features
        
        print(f'  Selected top {k} features by Mutual Information')
        print(f'  Top 10: {top_features[:10]}')
        
        # Plot MI scores
        fig, ax = plt.subplots(figsize=(12, 6))
        top_20 = mi_df.head(20)
        ax.barh(top_20['feature'], top_20['mi_score'], color='teal')
        ax.set_xlabel('Mutual Information Score')
        ax.set_title('Top 20 Features by Mutual Information')
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()
        
        keep_cols = top_features + [c for c in exclude if c in df.columns]
        return df[keep_cols]
    
    def scale_features(self, df: pd.DataFrame, method: str = 'robust') -> pd.DataFrame:
        """Scale numeric features using the specified method."""
        exclude = [self.target_binary, self.target_multi]
        feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
        
        if method == 'standard':
            self.scaler = StandardScaler()
        elif method == 'minmax':
            self.scaler = MinMaxScaler()
        elif method == 'robust':
            self.scaler = RobustScaler()
        
        df[feature_cols] = self.scaler.fit_transform(df[feature_cols])
        print(f'  Scaled {len(feature_cols)} features using {method} scaler')
        return df
    
    def process(self, df: pd.DataFrame, name: str, k_features: int = 50) -> pd.DataFrame:
        """Full feature engineering pipeline."""
        print(f'\n{"="*50}')
        print(f'  Feature Engineering: {name}')
        print(f'{"="*50}')
        
        df = self.encode_categoricals(df)
        df = self.remove_highly_correlated(df, threshold=0.95)
        df = self.select_features_mi(df, k=k_features)
        df = self.scale_features(df, method='robust')
        
        print(f'  Final shape: {df.shape}')
        return df

In [ ]:
# Process each dataset
fe_cicids = FeatureEngineer()
fe_unsw = FeatureEngineer()
fe_botiot = FeatureEngineer()

if not df_cicids.empty:
    df_cicids = fe_cicids.process(df_cicids, 'CICIDS2017', k_features=50)

if not df_unsw.empty:
    df_unsw = fe_unsw.process(df_unsw, 'UNSW-NB15', k_features=40)

if not df_botiot.empty:
    df_botiot = fe_botiot.process(df_botiot, 'Bot-IoT', k_features=30)

---
## 9. Class Balancing

Network intrusion datasets are heavily imbalanced. We use a combination of:
- **Undersampling** majority class
- **SMOTE** (Synthetic Minority Oversampling) — saved for model training notebooks
- Here we do **controlled undersampling** to avoid memory issues

In [ ]:
def balance_dataset(df: pd.DataFrame, target_col: str = 'is_attack',
                    max_samples_per_class: int = 100_000,
                    min_samples_per_class: int = 1000) -> pd.DataFrame:
    """
    Balance the dataset by capping majority classes and keeping minority classes.
    More sophisticated balancing (SMOTE) is done in training notebooks.
    """
    print(f'\nBalancing dataset (target: {target_col})...')
    print(f'  Before: {df[target_col].value_counts().to_dict()}')
    
    balanced_parts = []
    for cls in df[target_col].unique():
        cls_df = df[df[target_col] == cls]
        n = len(cls_df)
        
        if n > max_samples_per_class:
            cls_df = cls_df.sample(n=max_samples_per_class, random_state=RANDOM_STATE)
        elif n < min_samples_per_class:
            # Upsample very small classes
            cls_df = resample(cls_df, replace=True, n_samples=min_samples_per_class,
                            random_state=RANDOM_STATE)
        
        balanced_parts.append(cls_df)
    
    df_balanced = pd.concat(balanced_parts, ignore_index=True)
    df_balanced = df_balanced.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    
    print(f'  After:  {df_balanced[target_col].value_counts().to_dict()}')
    print(f'  Shape:  {df_balanced.shape}')
    return df_balanced

# Balance for binary classification
if not df_cicids.empty:
    df_cicids_balanced = balance_dataset(df_cicids, 'is_attack', max_samples_per_class=100_000)
else:
    df_cicids_balanced = pd.DataFrame()

if not df_unsw.empty:
    df_unsw_balanced = balance_dataset(df_unsw, 'is_attack', max_samples_per_class=80_000)
else:
    df_unsw_balanced = pd.DataFrame()

if not df_botiot.empty:
    df_botiot_balanced = balance_dataset(df_botiot, 'is_attack', max_samples_per_class=50_000)
else:
    df_botiot_balanced = pd.DataFrame()

---
## 10. Train / Validation / Test Split

Strategy: **70% Train / 15% Validation / 15% Test** with stratification.

In [ ]:
def split_dataset(df: pd.DataFrame, target_col: str = 'is_attack',
                  test_size: float = 0.15, val_size: float = 0.15,
                  random_state: int = 42) -> dict:
    """
    Split dataset into train/val/test with stratification.
    Returns dict with keys: X_train, X_val, X_test, y_train, y_val, y_test
    Also includes multi-class labels if 'attack_category' exists.
    """
    exclude_cols = ['is_attack', 'attack_category']
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    
    X = df[feature_cols].values.astype(np.float32)
    y_binary = df[target_col].values.astype(np.int64)
    
    # Check if multi-class labels exist
    y_multi = None
    multi_le = None
    if 'attack_category' in df.columns:
        multi_le = LabelEncoder()
        y_multi = multi_le.fit_transform(df['attack_category'].values)
    
    # First split: train+val vs test
    X_trainval, X_test, y_bin_trainval, y_bin_test = train_test_split(
        X, y_binary, test_size=test_size, random_state=random_state, stratify=y_binary
    )
    
    # Second split: train vs val
    val_fraction = val_size / (1 - test_size)
    X_train, X_val, y_bin_train, y_bin_val = train_test_split(
        X_trainval, y_bin_trainval, test_size=val_fraction,
        random_state=random_state, stratify=y_bin_trainval
    )
    
    result = {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train_binary': y_bin_train, 'y_val_binary': y_bin_val, 'y_test_binary': y_bin_test,
        'feature_names': feature_cols,
    }
    
    # Also split multi-class labels if available
    if y_multi is not None:
        y_multi_trainval, y_multi_test = train_test_split(
            y_multi, test_size=test_size, random_state=random_state, stratify=y_binary
        )[:2] if False else (None, None)
        
        # Re-split properly by using indices
        indices = np.arange(len(X))
        idx_trainval, idx_test = train_test_split(
            indices, test_size=test_size, random_state=random_state, stratify=y_binary
        )
        idx_train, idx_val = train_test_split(
            idx_trainval, test_size=val_fraction, random_state=random_state,
            stratify=y_binary[idx_trainval]
        )
        
        result['y_train_multi'] = y_multi[idx_train]
        result['y_val_multi'] = y_multi[idx_val]
        result['y_test_multi'] = y_multi[idx_test]
        result['multi_label_encoder'] = multi_le
        result['X_train'] = X[idx_train]
        result['X_val'] = X[idx_val]
        result['X_test'] = X[idx_test]
        result['y_train_binary'] = y_binary[idx_train]
        result['y_val_binary'] = y_binary[idx_val]
        result['y_test_binary'] = y_binary[idx_test]
    
    print(f'  Train: {result["X_train"].shape} | Val: {result["X_val"].shape} | Test: {result["X_test"].shape}')
    print(f'  Train label dist: {np.bincount(result["y_train_binary"])}')
    print(f'  Val label dist:   {np.bincount(result["y_val_binary"])}')
    print(f'  Test label dist:  {np.bincount(result["y_test_binary"])}')
    
    return result

# Split each dataset
splits = {}

print('\n--- CICIDS2017 Split ---')
if not df_cicids_balanced.empty:
    splits['cicids2017'] = split_dataset(df_cicids_balanced)

print('\n--- UNSW-NB15 Split ---')
if not df_unsw_balanced.empty:
    splits['unsw_nb15'] = split_dataset(df_unsw_balanced)

print('\n--- Bot-IoT Split ---')
if not df_botiot_balanced.empty:
    splits['bot_iot'] = split_dataset(df_botiot_balanced)

---
## 11. Create Unified Combined Dataset

Merge all datasets into a single unified dataset for cross-dataset training.
This requires aligning features across datasets.

In [ ]:
def create_unified_dataset(dataframes: dict, max_per_source: int = 50_000) -> pd.DataFrame:
    """
    Create a unified dataset from multiple sources.
    Finds common numeric features and combines with source tracking.
    """
    available = {k: v for k, v in dataframes.items() if not v.empty}
    
    if not available:
        print('[WARNING] No datasets available for unification.')
        return pd.DataFrame()
    
    # Find common numeric columns (excluding labels)
    exclude = ['is_attack', 'attack_category']
    
    all_numeric_cols = []
    for name, df in available.items():
        cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
        all_numeric_cols.append(set(cols))
    
    # If there are common features, use them; otherwise use all with NaN fill
    common_cols = set.intersection(*all_numeric_cols) if len(all_numeric_cols) > 1 else all_numeric_cols[0]
    common_cols = sorted(common_cols)
    
    print(f'Common numeric features across datasets: {len(common_cols)}')
    
    if len(common_cols) < 5:
        print('[INFO] Few common features. Using all features with NaN padding.')
        all_cols = set()
        for cols_set in all_numeric_cols:
            all_cols.update(cols_set)
        common_cols = sorted(all_cols)
    
    parts = []
    for name, df in available.items():
        # Sample if too large
        if len(df) > max_per_source:
            df_sample = df.sample(n=max_per_source, random_state=RANDOM_STATE)
        else:
            df_sample = df.copy()
        
        # Select columns (fill missing with 0)
        for col in common_cols:
            if col not in df_sample.columns:
                df_sample[col] = 0.0
        
        keep_cols = common_cols + [c for c in exclude if c in df_sample.columns]
        part = df_sample[keep_cols].copy()
        part['source_dataset'] = name
        parts.append(part)
        print(f'  {name}: {len(part):,} samples, {len(common_cols)} features')
    
    unified = pd.concat(parts, ignore_index=True)
    unified = unified.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    unified = unified.fillna(0)
    
    print(f'\nUnified dataset shape: {unified.shape}')
    print(f'Source distribution: {unified["source_dataset"].value_counts().to_dict()}')
    
    return unified

df_unified = create_unified_dataset({
    'cicids2017': df_cicids_balanced,
    'unsw_nb15': df_unsw_balanced,
    'bot_iot': df_botiot_balanced,
})

In [ ]:
# Split unified dataset
if not df_unified.empty:
    # Drop source_dataset column for ML
    df_unified_ml = df_unified.drop(columns=['source_dataset'], errors='ignore')
    
    print('--- Unified Dataset Split ---')
    splits['unified'] = split_dataset(df_unified_ml)

---
## 12. Save Processed Data & Artifacts

In [ ]:
def save_splits(splits_dict: dict, output_dir: Path):
    """Save all dataset splits as compressed NumPy files."""
    for dataset_name, split_data in splits_dict.items():
        dataset_dir = output_dir / dataset_name
        dataset_dir.mkdir(parents=True, exist_ok=True)
        
        # Save arrays
        for key in ['X_train', 'X_val', 'X_test',
                     'y_train_binary', 'y_val_binary', 'y_test_binary',
                     'y_train_multi', 'y_val_multi', 'y_test_multi']:
            if key in split_data and split_data[key] is not None:
                filepath = dataset_dir / f'{key}.npy'
                np.save(filepath, split_data[key])
                print(f'  Saved: {filepath} — shape {split_data[key].shape}')
        
        # Save feature names
        if 'feature_names' in split_data:
            filepath = dataset_dir / 'feature_names.npy'
            np.save(filepath, np.array(split_data['feature_names']))
            print(f'  Saved: {filepath}')
        
        # Save label encoder
        if 'multi_label_encoder' in split_data and split_data['multi_label_encoder'] is not None:
            filepath = dataset_dir / 'multi_label_encoder.joblib'
            joblib.dump(split_data['multi_label_encoder'], filepath)
            print(f'  Saved: {filepath}')
        
        print(f'  ✓ {dataset_name} saved to {dataset_dir}')
    
    print(f'\nAll splits saved to {output_dir}')

save_splits(splits, PROCESSED_DIR)

In [ ]:
# Save scalers and encoders for inference
artifacts_dir = MODELS_DIR / 'preprocessing'
artifacts_dir.mkdir(parents=True, exist_ok=True)

for name, fe in [('cicids2017', fe_cicids), ('unsw_nb15', fe_unsw), ('bot_iot', fe_botiot)]:
    if fe.scaler is not None:
        joblib.dump(fe.scaler, artifacts_dir / f'{name}_scaler.joblib')
        print(f'Saved scaler: {name}')
    
    if fe.label_encoders:
        joblib.dump(fe.label_encoders, artifacts_dir / f'{name}_label_encoders.joblib')
        print(f'Saved label encoders: {name}')
    
    if fe.selected_features is not None:
        joblib.dump(fe.selected_features, artifacts_dir / f'{name}_selected_features.joblib')
        print(f'Saved selected features: {name}')
    
    if fe.feature_importance is not None:
        fe.feature_importance.to_csv(artifacts_dir / f'{name}_feature_importance.csv', index=False)
        print(f'Saved feature importance: {name}')

print(f'\nAll preprocessing artifacts saved to {artifacts_dir}')

---
## 13. Verification & Summary

In [ ]:
# Verify saved files
print('\n' + '='*60)
print('  PREPROCESSING COMPLETE — FILE VERIFICATION')
print('='*60)

for dataset_name in splits.keys():
    dataset_dir = PROCESSED_DIR / dataset_name
    print(f'\n{dataset_name}/')
    if dataset_dir.exists():
        for f in sorted(dataset_dir.iterdir()):
            size_mb = f.stat().st_size / 1e6
            print(f'  {f.name:40s} {size_mb:8.2f} MB')
    else:
        print('  [NOT FOUND]')

print(f'\nArtifacts:')
if artifacts_dir.exists():
    for f in sorted(artifacts_dir.iterdir()):
        print(f'  {f.name}')

In [ ]:
# Quick load test
print('\n--- Quick Load Test ---')
for dataset_name in splits.keys():
    dataset_dir = PROCESSED_DIR / dataset_name
    try:
        X_train = np.load(dataset_dir / 'X_train.npy')
        y_train = np.load(dataset_dir / 'y_train_binary.npy')
        print(f'{dataset_name}: X_train={X_train.shape}, y_train={y_train.shape}, dtype={X_train.dtype}')
    except Exception as e:
        print(f'{dataset_name}: Load failed — {e}')

print('\n✓ Notebook 01 complete. Ready for Notebook 02 (Threat Detection).')

In [ ]:
# Free memory
del df_cicids, df_unsw, df_botiot
del df_cicids_balanced, df_unsw_balanced, df_botiot_balanced
del df_unified, df_unified_ml
gc.collect()
print('Memory freed.')